In [1]:
import random
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, DeepGraphInfomax

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score, f1_score

# ── Reproducibility ───────────────────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [2]:

# ── 1. Load flows and lookup ──────────────────────────────────────────────────
flows      = pd.read_csv("./data/ODWP01EW_OA.csv")
lookup_lsoa = pd.read_csv("./data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv")

flows['Output Areas code']    = flows['Output Areas code'].astype(str).str.strip()
flows['OA of workplace code'] = flows['OA of workplace code'].astype(str).str.strip()
lookup_lsoa['oa21cd']         = lookup_lsoa['oa21cd'].astype(str).str.strip()
lookup_lsoa['lsoa21cd']       = lookup_lsoa['lsoa21cd'].astype(str).str.strip()

oa_to_lsoa           = lookup_lsoa.set_index('oa21cd')['lsoa21cd'].to_dict()
flows['origin_lsoa']    = flows['Output Areas code'].map(oa_to_lsoa)
flows['workplace_lsoa'] = flows['OA of workplace code'].map(oa_to_lsoa)


C:\Users\jzvf437\AppData\Local\Temp\ipykernel_32684\2821207850.py:3: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup_lsoa = pd.read_csv("./data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv")


In [3]:
# ── 2. Filter to Leeds intra-city commutes ────────────────────────────────────
flows_complete = flows.dropna(subset=['origin_lsoa', 'workplace_lsoa']).copy()
leeds_lsoa     = lookup_lsoa[lookup_lsoa["ladnm"] == "Leeds"]["lsoa21cd"].unique()

leeds_flows = flows_complete[
    flows_complete["origin_lsoa"].isin(leeds_lsoa) &
    flows_complete["workplace_lsoa"].isin(leeds_lsoa)
].copy()

# Remove remote workers
leeds_flows = leeds_flows[
    leeds_flows["Place of work indicator (4 categories) label"]
    != "Mainly working at or from home, No fixed place"
].copy()

# Aggregate OD pairs
leeds_od = (
    leeds_flows
    .groupby(["origin_lsoa", "workplace_lsoa"])["Count"]
    .sum()
    .reset_index()
    .rename(columns={"Count": "flow"})
)

# Remove self-loops and log-transform
leeds_od = leeds_od[leeds_od['origin_lsoa'] != leeds_od['workplace_lsoa']].copy()
leeds_od['flow_log'] = np.log1p(leeds_od['flow'])

print(f"OD pairs after cleaning: {len(leeds_od)}")

OD pairs after cleaning: 54907


In [4]:
# ── 3. Node index — fixed and explicit ───────────────────────────────────────
# Sort for deterministic ordering
all_lsoas = sorted(
    pd.concat([leeds_od['origin_lsoa'], leeds_od['workplace_lsoa']]).unique()
)
lsoa_to_idx = {lsoa: i for i, lsoa in enumerate(all_lsoas)}
N = len(all_lsoas)
print(f"Unique LSOAs (nodes): {N}")

leeds_od['src'] = leeds_od['origin_lsoa'].map(lsoa_to_idx)
leeds_od['dst'] = leeds_od['workplace_lsoa'].map(lsoa_to_idx)


Unique LSOAs (nodes): 488


In [5]:
# ── 4. POI features ───────────────────────────────────────────────────────────
import geopandas as gpd

poi = gpd.read_file("./data/poi_uk.gpkg")
# Basic mapping from main_category to broader group
category_to_group = {

    # FOOD & DRINK
    'restaurant': 'FOOD_DRINK',
    'pub': 'FOOD_DRINK',
    'bar': 'FOOD_DRINK',
    'cafe': 'FOOD_DRINK',
    'coffee_shop': 'FOOD_DRINK',
    'fast_food': 'FOOD_DRINK',
    'fish_and_chips_restaurant': 'FOOD_DRINK',
    'chilean_restaurant': 'FOOD_DRINK',

    # SHOPPING / RETAIL
    'shopping': 'RETAIL',
    'shopping_mall': 'RETAIL',
    'grocery_or_supermarket': 'RETAIL',
    'convenience_store': 'RETAIL',
    'department_store': 'RETAIL',

    # LEISURE / SPORT
    'active_life': 'LEISURE_SPORT',
    'gym': 'LEISURE_SPORT',
    'sports_club': 'LEISURE_SPORT',
    'stadium': 'LEISURE_SPORT',
    'cinema': 'LEISURE_SPORT',
    'museum': 'LEISURE_SPORT',
    'tourist_attraction': 'LEISURE_SPORT',

    # BEAUTY & PERSONAL
    'beauty_and_spa': 'BEAUTY_PERSONAL',
    'hair_salon': 'BEAUTY_PERSONAL',
    'nail_salon': 'BEAUTY_PERSONAL',
    'spa': 'BEAUTY_PERSONAL',

    # PROFESSIONAL / BUSINESS
    'professional_services': 'PROFESSIONAL',
    'lawyer': 'PROFESSIONAL',
    'accountant': 'PROFESSIONAL',
    'consulting_agency': 'PROFESSIONAL',

    # HEALTH
    'hospital': 'HEALTH',
    'doctor': 'HEALTH',
    'dentist': 'HEALTH',
    'pharmacy': 'HEALTH',
    'elder_care_planning': 'HEALTH',

    # EDUCATION
    'school': 'EDUCATION',
    'primary_school': 'EDUCATION',
    'secondary_school': 'EDUCATION',
    'college': 'EDUCATION',
    'university': 'EDUCATION',
    'parenting_classes': 'EDUCATION',

    # AUTO
    'tire_repair_shop': 'AUTO',
    'car_repair': 'AUTO',
    'car_dealer': 'AUTO',
    'petrol_station': 'AUTO',

    # NATURE / OUTDOOR
    'park': 'NATURE_OUTDOOR',
    'sand_dune': 'NATURE_OUTDOOR',
    'beach': 'NATURE_OUTDOOR',
    'forest': 'NATURE_OUTDOOR',

    # RELIGION
    'church_cathedral': 'RELIGION',
    'church': 'RELIGION',
    'mosque': 'RELIGION',
    'synagogue': 'RELIGION',
    'temple': 'RELIGION',

    # ACCOMMODATION
    'hotel': 'ACCOMMODATION',
    'motel': 'ACCOMMODATION',
    'campground': 'ACCOMMODATION',
    'guest_house': 'ACCOMMODATION',
    'holiday_rental_home': 'ACCOMMODATION',
}


In [6]:
poi['poi_group'] = poi['main_category'].map(category_to_group)
poi_mapped = poi[poi['poi_group'].notna()].copy()

# Aggregate to LSOA level
poi_counts = (
    poi_mapped[poi_mapped['LSOA21CD'].isin(all_lsoas)]
    .groupby(['LSOA21CD', 'poi_group'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Build feature matrix — rows aligned to node index
feature_df = pd.DataFrame({'LSOA21CD': all_lsoas})  # sorted, deterministic
feature_df = feature_df.merge(poi_counts, on='LSOA21CD', how='left').fillna(0)

# Verify alignment
assert list(feature_df['LSOA21CD']) == all_lsoas, "Node order mismatch!"
assert len(feature_df) == N, "Row count mismatch!"

poi_cols = [c for c in feature_df.columns if c != 'LSOA21CD']

# Raw counts for reference, normalised for both models
poi_raw   = feature_df[poi_cols].values.copy()
poi_norm  = np.log1p(poi_raw)
poi_norm  = (poi_norm - poi_norm.mean(axis=0)) / (poi_norm.std(axis=0) + 1e-8)

print(f"Feature matrix: {feature_df.shape}")
print(f"LSOAs with all-zero POIs: {(poi_raw.sum(axis=1) == 0).sum()}")


Feature matrix: (488, 12)
LSOAs with all-zero POIs: 6


In [7]:

# ── 5. Labels — population and IMD ───────────────────────────────────────────
imd_data = pd.read_csv('./data/LSOA_IMD2025_OSGB1936_-7580747902701771840.csv')
pop_data = pd.read_csv('./data/new_population_data.csv')
pop_data['Total'] = pop_data['Total'].astype(str).str.replace(',', '').astype(float)

pop_df = (pop_data[pop_data['LSOA 2021 Code'].isin(all_lsoas)]
          [['LSOA 2021 Code', 'Total']]
          .rename(columns={'LSOA 2021 Code': 'LSOA21CD', 'Total': 'population'}))

imd_df = (imd_data[imd_data['LSOA Code'].isin(all_lsoas)]
          [['LSOA Code', 'IMD Decile']]
          .rename(columns={'LSOA Code': 'LSOA21CD', 'IMD Decile': 'imd_decile'}))

# Align to node index — same sorted order
eval_df = pd.DataFrame({'LSOA21CD': all_lsoas})
eval_df  = eval_df.merge(pop_df, on='LSOA21CD', how='left')
eval_df  = eval_df.merge(imd_df, on='LSOA21CD', how='left')
eval_df  = eval_df.fillna(0)

population = eval_df['population'].values
imd_decile = eval_df['imd_decile'].values.astype(int)

print(f"\nPopulation — matched: {(population>0).sum()}/{N}, "
      f"range: {population.min():.0f}–{population.max():.0f}")
print(f"IMD decile — matched: {(imd_decile>0).sum()}/{N}, "
      f"range: {imd_decile.min()}–{imd_decile.max()}")

# Sanity check alignment
check = pd.DataFrame({
    'LSOA21CD':  all_lsoas,
    'population': population,
    'imd_decile': imd_decile
})
print("\nAlignment check (first 5 rows):")
print(check.head())



Population — matched: 488/488, range: 988–4228
IMD decile — matched: 488/488, range: 1–10

Alignment check (first 5 rows):
    LSOA21CD  population  imd_decile
0  E01011264      1296.0           6
1  E01011265      1967.0           8
2  E01011266      2652.0          10
3  E01011267      1696.0           4
4  E01011268      1416.0           3


In [8]:
# Population
X_train, X_test, y_train, y_test = train_test_split(
    poi_norm, population, test_size=0.2, random_state=42)
grav_pop_r2 = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))
print(f"\nGravity; Population R²: {grav_pop_r2:.4f}")

# IMD
X_train, X_test, y_train, y_test = train_test_split(
    poi_norm, imd_decile, test_size=0.2, random_state=42)
grav_imd_acc = accuracy_score(y_test,
    RandomForestClassifier(n_estimators=100, random_state=42)
    .fit(X_train, y_train).predict(X_test))
print(f"Gravity; IMD Accuracy:  {grav_imd_acc:.4f}")


Gravity; Population R²: 0.0811
Gravity; IMD Accuracy:  0.1837


In [9]:

# ── 6. PyG graph ──────────────────────────────────────────────────────────────
x = torch.tensor(poi_norm, dtype=torch.float)

edge_index  = torch.tensor(leeds_od[['src', 'dst']].values.T, dtype=torch.long)
edge_weight = torch.tensor(leeds_od['flow_log'].values,        dtype=torch.float)

data = Data(x=x, edge_index=edge_index, edge_weight=edge_weight)

print(f"\nGraph — nodes: {data.num_nodes}, edges: {data.num_edges}")
print(f"x range: [{data.x.min():.3f}, {data.x.max():.3f}]")
print(f"Any NaN in x: {torch.isnan(data.x).any()}")

# ── 7. DGI encoder — with edge_weight ────────────────────────────────────────
class DGIEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels,   hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.prelu = torch.nn.PReLU()

    def forward(self, x, edge_index, edge_weight=None):
        x = self.conv1(x, edge_index, edge_weight)   # pass weights
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)   # pass weights
        x = self.prelu(x)
        return x

def corruption(x, edge_index, edge_weight=None):
    return x[torch.randperm(x.size(0))], edge_index, edge_weight

set_seed(42)

encoder = DGIEncoder(
    in_channels=data.x.shape[1],
    hidden_channels=128,
    out_channels=64
)

dgi = DeepGraphInfomax(
    hidden_channels=64,
    encoder=encoder,
    summary=lambda z, *args, **kwargs: torch.sigmoid(z.mean(dim=0)),
    corruption=corruption
)

optimizer = torch.optim.Adam(dgi.parameters(), lr=0.001)

# ── 8. Train DGI ──────────────────────────────────────────────────────────────
dgi.train()
for epoch in range(300):
    optimizer.zero_grad()
    pos_z, neg_z, summary = dgi(data.x, data.edge_index, data.edge_weight)  # pass weights
    loss = dgi.loss(pos_z, neg_z, summary)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")



Graph — nodes: 488, edges: 54907
x range: [-1.144, 6.580]
Any NaN in x: False
Epoch   0 | Loss: 1.3600
Epoch  50 | Loss: 0.0580
Epoch 100 | Loss: 0.1147
Epoch 150 | Loss: 0.0236
Epoch 200 | Loss: 0.0120
Epoch 250 | Loss: 0.0072


In [10]:

# ── 9. Extract embeddings ─────────────────────────────────────────────────────
dgi.eval()
with torch.no_grad():
    emb_tensor = encoder(data.x, data.edge_index, data.edge_weight)

emb = emb_tensor.numpy()
print(f"\nEmbeddings: {emb.shape}")
print(f"Range: [{emb.min():.3f}, {emb.max():.3f}]")
print(f"Std:   {emb.std():.4f}")
print(f"Collapsed dims (std < 0.01): {(emb.std(axis=0) < 0.01).sum()}")



Embeddings: (488, 64)
Range: [-0.327, 2.592]
Std:   0.2589
Collapsed dims (std < 0.01): 1


In [12]:
# ── Degree as a confounder ────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression as LR

degree = (np.bincount(leeds_od['src'].values, minlength=N) +
          np.bincount(leeds_od['dst'].values, minlength=N)).reshape(-1, 1)

degree_norm = (degree - degree.mean()) / (degree.std() + 1e-8)

# Residualise embeddings against degree
# i.e. remove the degree signal from every embedding dimension
residual_emb = emb.copy()
for i in range(emb.shape[1]):
    reg = LR().fit(degree_norm, emb[:, i])
    residual_emb[:, i] = emb[:, i] - reg.predict(degree_norm)

print("After residualising against degree:")
from scipy.stats import pearsonr
r_pop_resid    = np.mean([abs(pearsonr(residual_emb[:, i], population)[0])
                           for i in range(residual_emb.shape[1])])
r_degree_resid = np.mean([abs(pearsonr(residual_emb[:, i], degree_norm.ravel())[0])
                           for i in range(residual_emb.shape[1])])
print(f"  Mean abs correlation with population: {r_pop_resid:.3f}")
print(f"  Mean abs correlation with degree:     {r_degree_resid:.3f}")

# Evaluate residualised embeddings
dgi_resid_pop_r2, dgi_resid_imd_acc, dgi_resid_imd_f1 = evaluate(
    residual_emb, population, imd_decile, "DGI Residualised Embeddings")

After residualising against degree:
  Mean abs correlation with population: 0.135
  Mean abs correlation with degree:     0.000

DGI Residualised Embeddings
  Population R²:   -0.3280
  IMD Accuracy:    0.1429
  IMD Weighted F1: 0.1265


In [14]:

# ── 10. Downstream evaluation ─────────────────────────────────────────────────
def evaluate(X, population, imd_decile, label):
    # Population
    Xtr, Xte, ytr, yte = train_test_split(
        X, population, test_size=0.2, random_state=42)
    pop_r2 = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))

    # IMD
    Xtr, Xte, ytr, yte = train_test_split(
        X, imd_decile, test_size=0.2, random_state=42)
    rf       = RandomForestClassifier(n_estimators=100, random_state=42)
    imd_pred = rf.fit(Xtr, ytr).predict(Xte)
    imd_acc  = accuracy_score(yte, imd_pred)
    imd_f1   = f1_score(yte, imd_pred, average='weighted')

    print(f"\n{label}")
    print(f"  Population R²:   {pop_r2:.4f}")
    print(f"  IMD Accuracy:    {imd_acc:.4f}")
    print(f"  IMD Weighted F1: {imd_f1:.4f}")
    return pop_r2, imd_acc, imd_f1

feat_pop_r2, feat_imd_acc, feat_imd_f1 = evaluate(
    poi_norm, population, imd_decile, "Feature-based Model (normalised POI)")

dgi_pop_r2, dgi_imd_acc, dgi_imd_f1 = evaluate(
    emb, population, imd_decile, "DGI Embeddings")



Feature-based Model (normalised POI)
  Population R²:   0.0811
  IMD Accuracy:    0.1837
  IMD Weighted F1: 0.1685

DGI Embeddings
  Population R²:   -0.0742
  IMD Accuracy:    0.1939
  IMD Weighted F1: 0.1942


In [15]:
print(residual_emb.shape)
print(f"Mean abs correlation with population: {r_pop_resid:.3f}")
print(f"Mean abs correlation with degree:     {r_degree_resid:.3f}")

# Evaluate explicitly
Xtr, Xte, ytr, yte = train_test_split(
    residual_emb, population, test_size=0.2, random_state=42)
pop_r2 = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))

Xtr, Xte, ytr, yte = train_test_split(
    residual_emb, imd_decile, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
imd_acc = accuracy_score(yte, rf.fit(Xtr, ytr).predict(Xte))

print(f"\nResidualised DGI → Population R²: {pop_r2:.4f}")
print(f"Residualised DGI → IMD Accuracy:  {imd_acc:.4f}")

(488, 64)
Mean abs correlation with population: 0.135
Mean abs correlation with degree:     0.000

Residualised DGI → Population R²: -0.3280
Residualised DGI → IMD Accuracy:  0.1429


In [16]:
# Try multiple random seeds to see if the result is seed-dependent
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("DGI Population R² across different seeds:")
for seed in [0, 1, 7, 13, 21, 42, 99, 123, 200, 999]:
    Xtr, Xte, ytr, yte = train_test_split(
        emb, population, test_size=0.2, random_state=seed)
    r2 = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))
    print(f"  seed={seed:4d}  R²={r2:.4f}")

print("\nFeature-based Population R² across different seeds:")
for seed in [0, 1, 7, 13, 21, 42, 99, 123, 200, 999]:
    Xtr, Xte, ytr, yte = train_test_split(
        poi_norm, population, test_size=0.2, random_state=seed)
    r2 = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))
    print(f"  seed={seed:4d}  R²={r2:.4f}")

DGI Population R² across different seeds:
  seed=   0  R²=-0.0642
  seed=   1  R²=-0.7718
  seed=   7  R²=-0.3239
  seed=  13  R²=-0.1930
  seed=  21  R²=-0.2143
  seed=  42  R²=-0.0742
  seed=  99  R²=-0.1862
  seed= 123  R²=-0.2688
  seed= 200  R²=-0.0459
  seed= 999  R²=-0.4255

Feature-based Population R² across different seeds:
  seed=   0  R²=-0.0336
  seed=   1  R²=-0.2050
  seed=   7  R²=0.1456
  seed=  13  R²=0.0587
  seed=  21  R²=0.1196
  seed=  42  R²=0.0811
  seed=  99  R²=-0.0475
  seed= 123  R²=-0.2410
  seed= 200  R²=0.1452
  seed= 999  R²=0.1591


In [ ]:

# ── 11. Flow prediction ───────────────────────────────────────────────────────
lsoa_centroids = pd.read_csv(
    './data/LSOA_PopCentroids_EW_2021_V4_-1224685956027462334.csv')

centroid_df = pd.DataFrame({'LSOA21CD': all_lsoas})
centroid_df = centroid_df.merge(
    lsoa_centroids[['LSOA21CD', 'x', 'y']], on='LSOA21CD', how='left')

print(f"\nCentroids matched: {centroid_df['x'].notna().sum()} / {N}")

coords  = torch.tensor(centroid_df[['x', 'y']].values, dtype=torch.float)
src_idx = torch.tensor(leeds_od['src'].values, dtype=torch.long)
dst_idx = torch.tensor(leeds_od['dst'].values, dtype=torch.long)

dist     = torch.sqrt(((coords[src_idx] - coords[dst_idx]) ** 2).sum(dim=1))
dist_log = torch.log1p(dist)
dist_log = (dist_log - dist_log.mean()) / (dist_log.std() + 1e-8)

# DGI flow features
od_emb = torch.cat([
    emb_tensor[src_idx],
    emb_tensor[dst_idx],
    dist_log.unsqueeze(1)
], dim=1).detach().numpy()

# Gravity flow features (population + distance)
pop_tensor = torch.tensor(population, dtype=torch.float)
od_gravity = torch.cat([
    pop_tensor[src_idx].unsqueeze(1),
    pop_tensor[dst_idx].unsqueeze(1),
    dist_log.unsqueeze(1)
], dim=1).numpy()

flow_target = leeds_od['flow_log'].values

Xtr, Xte, ytr, yte = train_test_split(
    od_emb, flow_target, test_size=0.2, random_state=42)
dgi_flow_r2 = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))

Xtr, Xte, ytr, yte = train_test_split(
    od_gravity, flow_target, test_size=0.2, random_state=42)
grav_flow_r2 = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))

print(f"\nFlow Prediction")
print(f"  DGI + Distance:      R² = {dgi_flow_r2:.4f}")
print(f"  Gravity (Pop+Dist):  R² = {grav_flow_r2:.4f}")

# ── 12. Summary table ─────────────────────────────────────────────────────────
print("\n" + "="*55)
print("LEEDS RESULTS SUMMARY")
print("="*55)
print(f"{'Model':<30} {'Pop R²':>8} {'IMD Acc':>9} {'IMD F1':>8}")
print("-"*55)
print(f"{'Feature-based (norm POI)':<30} {feat_pop_r2:>8.4f} "
      f"{feat_imd_acc:>9.4f} {feat_imd_f1:>8.4f}")
print(f"{'DGI':<30} {dgi_pop_r2:>8.4f} "
      f"{dgi_imd_acc:>9.4f} {dgi_imd_f1:>8.4f}")
print(f"{'Flow — DGI+Dist':<30} {'':>8} {'':>9} {dgi_flow_r2:>8.4f}")
print(f"{'Flow — Gravity':<30} {'':>8} {'':>9} {grav_flow_r2:>8.4f}")
print("="*55)
The two fixes that matter most are passing edge_weight through the encoder and using a sorted, deterministic node ordering instead of pd.concat().unique() which can vary. Run this clean from top to bottom and report back the results. Sonnet 4.6